## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)
print("TensorFlow", tf.__version__, "| GPU:", gpus)

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
print("TensorFlow version:", tf.__version__)

## 2. Loading and explore the dataset

In [ ]:
import os
from tensorflow.keras.preprocessing import image
import matplotlib.pyplot as plt

# Define the dataset path
dataset_path = r"D:\Rice_DL_Project\Rice_Classification\data\raw\Rice_Image_Dataset"  

# Explore the dataset
categories = [category for category in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, category))]
print("Categories:", categories)

# Visualize some sample images from each category
for category in categories:
    category_path = os.path.join(dataset_path, category)
    sample_image = os.listdir(category_path)[0]  # Pick one image from each category
    img = image.load_img(os.path.join(category_path, sample_image), target_size=(224, 224))
    plt.imshow(img)
    plt.title(f"Category: {category}")
    plt.show()

## 3. Data quality check

Same as Model 1: check class balance, corrupt files, image size/mode and duplicate files (MD5). Duplicates are removed before splitting to avoid train/test leakage.

In [ ]:
from PIL import Image
import hashlib

rows = []
for category in sorted(categories):
    folder = os.path.join(dataset_path, category)
    for fname in sorted(os.listdir(folder)):
        path = os.path.join(folder, fname)
        try:
            with Image.open(path) as im:
                im.verify()                      # detect corrupt files
            with Image.open(path) as im:
                size, mode = im.size, im.mode
            with open(path, "rb") as f:
                md5 = hashlib.md5(f.read()).hexdigest()
            rows.append((path, category, size, mode, md5))
        except Exception as e:
            print("Corrupt:", path, e)

info = pd.DataFrame(rows, columns=["filepath", "label", "size", "mode", "md5"])
print("Images per class:\n", info["label"].value_counts(), "\n")
print("Image sizes:\n", info["size"].value_counts(), "\n")
print("Color modes:\n", info["mode"].value_counts(), "\n")
print("Duplicate files:", info["md5"].duplicated().sum())
print("Duplicate groups with conflicting labels:", (info.groupby("md5")["label"].nunique() > 1).sum())

# Drop conflicting-label groups, then keep one copy of each duplicate
conflict = info.groupby("md5")["label"].transform("nunique") > 1
clean_df = info[~conflict].drop_duplicates("md5").reset_index(drop=True)
print(f"Kept {len(clean_df)} of {len(info)} images")

## 4. Data Preprocessing

Same split as Model 1 (70/15/15, stratified, `random_state=42`), so the test set is identical across models.
Augmentation (flips + rotation) is applied to the **train** generator only. Val/test only rescale and are not shuffled.

In [ ]:
# Use the cleaned (filepath, label) table from the data quality check
df = clean_df[["filepath", "label"]]

# Split 70% train, 15% validation, 15% test
train_df, temp_df = train_test_split(df, test_size=0.3, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, stratify=temp_df["label"], random_state=42)
print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

# Augmentation only for train. No brightness/color/zoom changes: color separates rice varieties.
train_datagen = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    vertical_flip=True,
    rotation_range=20,
)
eval_datagen = ImageDataGenerator(rescale=1./255)   # val and test

def make_generator(data, datagen, shuffle):
    return datagen.flow_from_dataframe(
        data,
        x_col="filepath",
        y_col="label",
        target_size=(224, 224),
        batch_size=32,
        class_mode="categorical",
        shuffle=shuffle,
    )

train_generator = make_generator(train_df, train_datagen, shuffle=True)
validation_generator = make_generator(val_df, eval_datagen, shuffle=False)
test_generator = make_generator(test_df, eval_datagen, shuffle=False)

## 5. CNN model

Repeated CNN block: `[Conv3x3 -> BatchNorm -> ReLU] x2 -> MaxPool(2)`, called 4 times with 32/64/128/256 filters.
GlobalAveragePooling replaces Flatten + hidden Dense.

In [ ]:
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

def conv_block(x, filters):
    for _ in range(2):
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    return layers.MaxPooling2D(2)(x)

inputs = Input(shape=(224, 224, 3))
x = conv_block(inputs, 32)    # 112x112x32
x = conv_block(x, 64)         # 56x56x64
x = conv_block(x, 128)        # 28x28x128
x = conv_block(x, 256)        # 14x14x256
x = layers.GlobalAveragePooling2D()(x)   # 256
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(5, activation="softmax")(x)

model = Model(inputs, outputs, name="cnn_blocks")

## 6. Compiling the CNN Model

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 7. Model Training

Train with EarlyStopping (max 20 epochs) and save to disk. If both saved files exist the model is loaded instead of retrained (set `FORCE_RETRAIN = True` to retrain).

In [ ]:
import json

FORCE_RETRAIN = False
models_dir = r"D:\Rice_DL_Project\Rice_Classification\models"
os.makedirs(models_dir, exist_ok=True)
model_path = os.path.join(models_dir, "model2_cnn_blocks.h5")
history_path = os.path.join(models_dir, "model2_cnn_blocks_history.json")

if os.path.exists(model_path) and os.path.exists(history_path) and not FORCE_RETRAIN:
    # Reuse the saved model and its training history
    model = tf.keras.models.load_model(model_path, compile=False)
    with open(history_path) as f:
        history_dict = json.load(f)
    print("Loaded saved model:", model_path)
else:
    early_stop = EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)
    history = model.fit(
        train_generator,
        epochs=20,
        validation_data=validation_generator,
        callbacks=[early_stop],
    )
    history_dict = history.history
    model.save(model_path, include_optimizer=False)
    with open(history_path, "w") as f:
        json.dump(history_dict, f)
    print("Trained and saved model to:", model_path)

print("Epochs run:", len(history_dict["loss"]))
print("Final epoch:", {k: round(v[-1], 4) for k, v in history_dict.items()})

## 8. Plots

In [ ]:
import matplotlib.pyplot as plt

# Plot training & validation accuracy values
plt.plot(history_dict['accuracy'])
plt.plot(history_dict['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

# Plot training & validation loss values
plt.plot(history_dict['loss'])
plt.plot(history_dict['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Train', 'Validation'], loc='upper left')
plt.show()

## 9. Model Evaluation

Evaluated on the held-out test set (same as Model 1). Precision, recall and F1 are macro-averaged.

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

class_names = list(test_generator.class_indices.keys())
y_pred = np.argmax(model.predict(test_generator), axis=1)
y_true = test_generator.classes

print(f"Accuracy : {accuracy_score(y_true, y_pred):.4f}")
print(f"Precision: {precision_score(y_true, y_pred, average='macro'):.4f}")
print(f"Recall   : {recall_score(y_true, y_pred, average='macro'):.4f}")
print(f"F1-score : {f1_score(y_true, y_pred, average='macro'):.4f}")
print()
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## 10. Model Prediction

In [ ]:
X_test, y_test = test_generator.next() 
predictions = model.predict(X_test)

def plot_predictions(X_test, y_test, predictions, class_indices):
    plt.figure(figsize=(12, 12))
    for i in range(9):
        plt.subplot(3, 3, i + 1)
        plt.imshow(X_test[i])
        true_label = np.argmax(y_test[i])
        predicted_label = np.argmax(predictions[i])
        true_class = list(class_indices.keys())[list(class_indices.values()).index(true_label)]
        predicted_class = list(class_indices.keys())[list(class_indices.values()).index(predicted_label)]
        plt.title(f"True: {true_class}\nPredicted: {predicted_class}")
        plt.axis('off')
    plt.tight_layout()
    plt.show()

plot_predictions(X_test, y_test, predictions, test_generator.class_indices)